# Game Recommendation System
## Exploratory Data Analysis + Model Training + Evaluation

**Approaches**:
1. Content-Based Filtering (TF-IDF + cosine similarity)
2. Collaborative Filtering (user-item matrix + cosine similarity)

**Metrics**: precision@k, recall@k, RMSE

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
from utils import load_data

DATA_PATH = os.path.join('..', 'data', 'steam_games.csv')

if not os.path.exists(DATA_PATH):
    print('Dataset not found. Generating synthetic data...')
    rng = np.random.default_rng(42)
    genres_pool = ['Action','RPG','Strategy','Adventure','Indie','Simulation','Horror','Puzzle']
    rows = []
    for i in range(500):
        g = ', '.join(rng.choice(genres_pool, size=rng.integers(1, 4), replace=False).tolist())
        rows.append({
            'name': f'Game {i}',
            'genres': g,
            'steamspy_tags': g.replace(', ', ' '),
            'short_description': f'An exciting {g} game with unique mechanics.',
            'rating': round(rng.uniform(1, 5), 2),
        })
    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
else:
    df = load_data(DATA_PATH)

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)

In [ ]:
# Rating distribution
if 'rating' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df['rating'].dropna(), bins=30, ax=ax)
    ax.set_title('Rating Distribution')
    ax.set_xlabel('Rating')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top genres
if 'genres' in df.columns:
    genre_series = df['genres'].dropna().str.split(', ').explode()
    top_genres = genre_series.value_counts().head(15)

    fig, ax = plt.subplots(figsize=(10, 5))
    top_genres.plot(kind='bar', ax=ax)
    ax.set_title('Top 15 Genres')
    ax.set_xlabel('Genre')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 3. Content-Based Filtering

In [ ]:
from content_based import build_tfidf_matrix, get_recommendations as cb_recommend

tfidf_matrix, vectorizer = build_tfidf_matrix(df)
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

In [ ]:
# Test a recommendation
query_game = df['name'].iloc[0]
recs_cb = cb_recommend(query_game, df, tfidf_matrix, top_n=10)
print(f"Content-Based recommendations for '{query_game}':")
recs_cb

## 4. Collaborative Filtering

In [ ]:
from collaborative import build_user_item_matrix, get_recommendations as cf_recommend, evaluate

# Generate synthetic interactions
rng = np.random.default_rng(42)
game_names = df['name'].tolist()
users = [f'User_{i}' for i in range(200)]

interaction_rows = []
for u in users:
    for g in rng.choice(game_names, size=rng.integers(5, 25), replace=False):
        interaction_rows.append({'user_id': u, 'game_name': g, 'rating': int(rng.integers(1, 6))})

interactions = pd.DataFrame(interaction_rows)
print(f'Interactions shape: {interactions.shape}')
interactions.head()

In [ ]:
# Train/test split by user
from sklearn.model_selection import train_test_split

train_interactions, test_interactions = train_test_split(
    interactions, test_size=0.2, random_state=42
)

user_item_matrix = build_user_item_matrix(train_interactions)
print(f'User-item matrix shape: {user_item_matrix.shape}')

In [ ]:
# Test a recommendation
query_user = users[0]
recs_cf = cf_recommend(query_user, user_item_matrix, top_n=10)
print(f"Collaborative Filtering recommendations for '{query_user}':")
recs_cf

## 5. Evaluation & Metrics

In [ ]:
metrics_cf = evaluate(user_item_matrix, test_interactions, k=10)

print('=== Collaborative Filtering Metrics ===')
for metric, value in metrics_cf.items():
    print(f'  {metric}: {value}')

In [ ]:
# Compare models visually
from utils import precision_at_k, recall_at_k

# Simulate content-based metrics (based on genre overlap as proxy for relevance)
cb_precisions, cb_recalls = [], []
for _ in range(50):
    game = df['name'].sample(1, random_state=rng.integers(1000)).iloc[0]
    try:
        rec = cb_recommend(game, df, tfidf_matrix, top_n=10)
        ref_genre = df[df['name'] == game]['genres'].iloc[0]
        relevant = df[df['genres'] == ref_genre]['name'].tolist()
        cb_precisions.append(precision_at_k(rec['name'].tolist(), relevant, 10))
        cb_recalls.append(recall_at_k(rec['name'].tolist(), relevant, 10))
    except Exception:
        continue

results = pd.DataFrame({
    'Model': ['Content-Based', 'Collaborative Filtering'],
    'Precision@10': [round(float(np.mean(cb_precisions)), 4), metrics_cf.get('precision@10', 0)],
    'Recall@10': [round(float(np.mean(cb_recalls)), 4), metrics_cf.get('recall@10', 0)],
})

print('\n=== Model Comparison ===')
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
results.plot(x='Model', y='Precision@10', kind='bar', ax=axes[0], legend=False, color=['steelblue', 'salmon'])
axes[0].set_title('Precision@10')
axes[0].set_ylabel('Score')
axes[0].tick_params(axis='x', rotation=20)

results.plot(x='Model', y='Recall@10', kind='bar', ax=axes[1], legend=False, color=['steelblue', 'salmon'])
axes[1].set_title('Recall@10')
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=120)
plt.show()

## 6. Summary

| Model | Precision@10 | Recall@10 |
|---|---|---|
| Content-Based | — | — |
| Collaborative Filtering | — | — |

> Replace the `—` values with results from cell above after running.

**Next step**: Run `streamlit run app/streamlit_app.py` to test the interactive demo.